# Phase 10.2: Needleman-Wunsch CUDA Longer Sequence Support


This notebook explores longer-sequence Needleman-Wunsch CUDA support using global-memory DP matrices, rolling diagonals, and an experimental tiled-wavefront prototype.

In [ ]:
!nvidia-smi
!nvcc --version


In [ ]:
import os
print("Current working directory:", os.getcwd())


Compile the long-sequence CUDA implementation.

In [ ]:
!nvcc src/needleman_wunsch_gpu_longseq.cu \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o needleman_wunsch_gpu_longseq


Generate fixed-length synthetic datasets for longer-sequence validation.

In [ ]:
!python scripts/generate_synthetic_dataset.py \
  --num-pairs 100 \
  --sequence-length 128 \
  --output data/synthetic/synthetic_pairs_128.txt \
  --seed 42

!python scripts/generate_synthetic_dataset.py \
  --num-pairs 100 \
  --sequence-length 256 \
  --output data/synthetic/synthetic_pairs_256.txt \
  --seed 42


Run the rolling-diagonal implementation.

In [ ]:
!mkdir -p results/needleman_wunsch

!./needleman_wunsch_gpu_longseq \
  data/synthetic/synthetic_pairs_128.txt \
  results/needleman_wunsch/needleman_wunsch_gpu_longseq_rolling_results.csv \
  --implementation rolling_diagonal \
  --repetitions 3 \
  --summary-only


Run the global-matrix implementation.

In [ ]:
!./needleman_wunsch_gpu_longseq \
  data/synthetic/synthetic_pairs_128.txt \
  results/needleman_wunsch/needleman_wunsch_gpu_longseq_global_results.csv \
  --implementation global_matrix \
  --repetitions 3 \
  --summary-only


Optionally run the experimental tiled-wavefront prototype.

In [ ]:
!./needleman_wunsch_gpu_longseq \
  data/synthetic/synthetic_pairs_128.txt \
  results/needleman_wunsch/needleman_wunsch_gpu_longseq_tiled_results.csv \
  --implementation tiled_wavefront \
  --tile-size 16 \
  --repetitions 3 \
  --summary-only


Run the quick long-sequence benchmark and generate charts.

In [ ]:
!python benchmarks/run_needleman_wunsch_longseq_benchmark.py --quick


In [ ]:
!python scripts/plot_needleman_wunsch_longseq_benchmark.py


Display benchmark results.

In [ ]:
import pandas as pd
df = pd.read_csv("benchmarks/needleman_wunsch_longseq_benchmark_results.csv")
df


Display generated charts.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

chart_directory = Path("assets/benchmark_charts/needleman_wunsch_longseq")
for chart_path in sorted(chart_directory.glob("*.png")):
    print(chart_path)
    display(Image(filename=str(chart_path)))
